In [8]:
import asyncio
import json
import websockets
import pandas as pd
from datetime import datetime

symbol = "btcusdt"  # change to ethusdt, solusdt etc
url = "wss://fstream.binance.com/ws/btcusdt@trade"

# ----------------------------
# CONFIG (tweak this)
# ----------------------------
WHALE_USDT_THRESHOLD = 100000  # $100k per trade
VOLUME_WINDOW = 20

trade_history = []

# ----------------------------
# WHALE DETECTOR
# ----------------------------
def detect_whale(trade):
    price = float(trade['p'])
    qty = float(trade['q'])
    value = price * qty

    is_buyer_maker = trade['m']  # True = sell aggressor, False = buy aggressor

    side = "SELL" if is_buyer_maker else "BUY"

    return value, side

# ----------------------------
# MAIN LOOP
# ----------------------------
async def run():
    async with websockets.connect(url) as ws:
        print("🚀 Whale Bot Running...")

        while True:
            msg = await ws.recv()
            trade = json.loads(msg)

            value, side = detect_whale(trade)

            # store trade
            trade_history.append(value)
            if len(trade_history) > VOLUME_WINDOW:
                trade_history.pop(0)

            avg_volume = sum(trade_history) / len(trade_history)

            time = datetime.fromtimestamp(trade['T']/1000)

            # ----------------------------
            # WHALE CONDITIONS
            # ----------------------------
            if value >= WHALE_USDT_THRESHOLD:
                print(f"\n🐋 WHALE TRADE DETECTED")
                print(f"Time: {time}")
                print(f"Side: {side}")
                print(f"Value: ${value:,.2f}")

            # volume spike detection
            elif value > avg_volume * 5:
                print(f"\n⚡ VOLUME SPIKE")
                print(f"Time: {time}")
                print(f"Side: {side}")
                print(f"Value: ${value:,.2f} (Avg: ${avg_volume:,.2f})")

# run bot
await run()

🚀 Whale Bot Running...

🐋 WHALE TRADE DETECTED
Time: 2026-05-30 10:37:32.017000
Side: BUY
Value: $112,839.68

⚡ VOLUME SPIKE
Time: 2026-05-30 10:37:32.017000
Side: BUY
Value: $80,452.56 (Avg: $12,921.73)

⚡ VOLUME SPIKE
Time: 2026-05-30 10:37:38.969000
Side: BUY
Value: $11,482.77 (Avg: $1,335.98)

⚡ VOLUME SPIKE
Time: 2026-05-30 10:37:38.969000
Side: BUY
Value: $11,188.34 (Avg: $2,171.42)

🐋 WHALE TRADE DETECTED
Time: 2026-05-30 10:37:38.969000
Side: BUY
Value: $112,840.30

⚡ VOLUME SPIKE
Time: 2026-05-30 10:37:38.969000
Side: BUY
Value: $72,429.78 (Avg: $11,457.01)

🐋 WHALE TRADE DETECTED
Time: 2026-05-30 10:37:38.969000
Side: BUY
Value: $119,906.62

⚡ VOLUME SPIKE
Time: 2026-05-30 10:37:45.632000
Side: BUY
Value: $25,026.79 (Avg: $3,838.67)

⚡ VOLUME SPIKE
Time: 2026-05-30 10:37:45.632000
Side: BUY
Value: $57,046.35 (Avg: $6,444.40)

⚡ VOLUME SPIKE
Time: 2026-05-30 10:38:02.190000
Side: BUY
Value: $27,603.07 (Avg: $2,447.47)

⚡ VOLUME SPIKE
Time: 2026-05-30 10:38:18.643000
Side: SELL

CancelledError: 

In [9]:
import asyncio
import websockets
import json
from collections import deque

symbol = "btcusdt"

trade_url = f"wss://fstream.binance.com/ws/{symbol}@trade"
depth_url = f"wss://fstream.binance.com/ws/{symbol}@depth@100ms"

# ---------------------------
# STORAGE
# ---------------------------
trade_buffer = deque(maxlen=100)
bid_volume = 0
ask_volume = 0

# ---------------------------
# TRADE STREAM (MARKET ORDERS)
# ---------------------------
async def trade_stream():
    async with websockets.connect(trade_url) as ws:
        print("🔥 Trade stream connected")

        while True:
            msg = json.loads(await ws.recv())

            price = float(msg['p'])
            qty = float(msg['q'])
            value = price * qty

            is_sell = msg['m']  # True = sell aggressor

            side = "SELL" if is_sell else "BUY"

            trade_buffer.append((side, value))

            print(f"TRADE | {side} | ${value:,.0f}")

# ---------------------------
# ORDER BOOK STREAM (LIMIT ORDERS)
# ---------------------------
async def depth_stream():
    global bid_volume, ask_volume

    async with websockets.connect(depth_url) as ws:
        print("📦 Depth stream connected")

        while True:
            msg = json.loads(await ws.recv())

            bids = msg['b'][:10]  # top 10 levels
            asks = msg['a'][:10]

            bid_volume = sum(float(b[1]) for b in bids)
            ask_volume = sum(float(a[1]) for a in asks)

            imbalance = (bid_volume - ask_volume) / (bid_volume + ask_volume + 1e-9)

            print(f"ORDERBOOK | Imbalance: {imbalance:.2f}")

            # Liquidity pressure detection
            if imbalance > 0.3:
                print("🟢 BUY SIDE LIQUIDITY STRONG")
            elif imbalance < -0.3:
                print("🔴 SELL SIDE LIQUIDITY STRONG")

# ---------------------------
# RUN BOTH STREAMS
# ---------------------------
async def main():
    await asyncio.gather(
        trade_stream(),
        depth_stream()
    )

await main()

📦 Depth stream connected
🔥 Trade stream connected
ORDERBOOK | Imbalance: 0.78
🟢 BUY SIDE LIQUIDITY STRONG
ORDERBOOK | Imbalance: 0.26
TRADE | BUY | $147
ORDERBOOK | Imbalance: 0.31
🟢 BUY SIDE LIQUIDITY STRONG
ORDERBOOK | Imbalance: 0.67
🟢 BUY SIDE LIQUIDITY STRONG
ORDERBOOK | Imbalance: 0.05
ORDERBOOK | Imbalance: -0.41
🔴 SELL SIDE LIQUIDITY STRONG
TRADE | BUY | $10,894
TRADE | BUY | $29,444
ORDERBOOK | Imbalance: 0.66
🟢 BUY SIDE LIQUIDITY STRONG
ORDERBOOK | Imbalance: 0.68
🟢 BUY SIDE LIQUIDITY STRONG
ORDERBOOK | Imbalance: 0.66
🟢 BUY SIDE LIQUIDITY STRONG
ORDERBOOK | Imbalance: 0.65
🟢 BUY SIDE LIQUIDITY STRONG
ORDERBOOK | Imbalance: 0.52
🟢 BUY SIDE LIQUIDITY STRONG
ORDERBOOK | Imbalance: 0.68
🟢 BUY SIDE LIQUIDITY STRONG
TRADE | SELL | $589
ORDERBOOK | Imbalance: 0.91
🟢 BUY SIDE LIQUIDITY STRONG
ORDERBOOK | Imbalance: 0.52
🟢 BUY SIDE LIQUIDITY STRONG
TRADE | BUY | $0
ORDERBOOK | Imbalance: 0.87
🟢 BUY SIDE LIQUIDITY STRONG
ORDERBOOK | Imbalance: 0.87
🟢 BUY SIDE LIQUIDITY STRONG
ORDERBOO

CancelledError: 